In [1]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

/home/gtamo/MS_ML


In [2]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm.auto import tqdm  # auto -> notebook/VSCode widget
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import pyarrow.dataset as pads
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date

# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
from get_library import get_df   # CDD Vault collection export
from python.MS_build_ML import PARAMS, DATA, OUTPUT   # CLI classes reused here (%autoreload picks up edits)
# from tdc.multi_pred import DTI

/home/gtamo/miniconda3/envs/ML/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
## params — every YAML key becomes an attribute (params.MS_PATH); Dropbox readers + MLTrail vault bound here
CONFIG = 'config/config.yaml'

params = PARAMS(CONFIG)
params.load_params()
params.setup_dropbox()    # params.dbx / params.dbx_glob per DROPBOX_BACKEND
params.load_registry()    # params.registry (shared MLTrail vault)

## transition shim — keeps the bare YAML names (SERAC_C, MS_PATH, ...) working in the cells below
globals().update({k: v for k, v in params.__dict__.items() if k.isupper()})
registry = params.registry
_dbx = _fbx_open = params.dbx
_fbx_glob = params.dbx_glob

print('> chemlib overwrite:', params.CHEMLIB_OVERWRITE, '| dfraw overwrite:', params.DFRAW_OVERWRITE)
print('> features:', params.FEATURES_TYPE)

> loaded 58 params from config/config.yaml
> Dropbox backend: rclone
> chemlib overwrite: False | dfraw overwrite: False
> features: H237


## 0. Imports

In [4]:
%%time
## data — library + proteomics + chemistry features (see python/MS_build_ML.py:DATA)
data = DATA()
data.load_chemical_lib_df(params)                                  # serac_df + validated / devalidated targets
## cm2rm now comes from config CM2RM_PARTS — any subset of contaminants (the CONTAMINANTS csv) |
## control_compounds (CONTROLS) | fbx_independent (Px_Ligase_dependent == 0 or Px_validated_WT == 0)
data.get_contaminants_and_controls(params)                         # parts=(...) here would override the config
data.load_proteomics_data(params)                                  # DFRAW_OVERWRITE -> rebuild from the tranches, else read the parquets
data.compute_features(params)                                      # FEATURES_TYPE -> MF_features (+ MF morgan bits)

## OpenTargets disease scores + the big-pharma-relevant subset (see python/MS_build_ML.py:DATA)
data.load_opentargets(params)   # ot_df, cached to OT_CACHE — delete that file to force a recompute from OT_ROOT
data.filter_pharma_targets()    # ot_pharma, via the module-level PHARMA_PRIORITY_AREAS / PHARMA_DROP_AREAS

## transition shim — bare names used by the cells below

## transition shim — bare names used by the cells below
serac_df, MS, df_raw, MF, MF_features = data.serac_df, data.MS, data.df_raw, data.MF, data.MF_features
control_compounds, contaminants, cm2rm = data.control_compounds, data.contaminants, data.cm2rm
fbx_independent = data.fbx_independent
validated_targets, devalidated_targets = data.validated_targets, data.devalidated_targets
ot_df, ot_pharma = data.ot_df, data.ot_pharma
# ot_df.to_csv('data/patent/20260609_OpenTarget_list.csv', sep=',', index=False)

> Chemical lib dim: (11197, 11) | 30 validated / 1172 devalidated targets
> 2 control + 81 contaminant compounds (from config) + 594 fbx independent | cm2rm = contaminants + fbx_independent = 675
> loaded cached MS (6707, 5) | df_raw (60971034, 12)
> EXCLUDE_DATES ['2026-08-13']: dropped 17,212,863 df_raw rows / 1,848 MS compounds
> checking smiles


100%|██████████| 11197/11197 [00:02<00:00, 4035.49it/s]


> FEATURES_TYPE: H237


H237 descriptors: 100%|██████████| 128/128 [00:06<00:00, 18.30it/s]


> 0 smiles failed | MF (11197, 2049) | MF_features (11149, 4470)
> loaded cached output/MS/opentargets_target_disease.parquet
> 232,004 (target, disease) rows / 11,584 targets resolved
> 78,935 / 232,004 (target, disease) rows pass the pharma filter | 9,105 unique targets
  17,071  cancer or benign tumor
   3,890  cardiovascular disease
   2,504  disorder of visual system
   3,466  endocrine system disease
   3,416  hematologic disease
   3,938  immune system disease
   3,710  infectious disease
   8,708  musculoskeletal or connective tissue disease
  25,393  nervous system disease
   9,849  nutritional or metabolic disease
   1,704  psychiatric disorder
   2,357  respiratory or thoracic disease
CPU times: user 41.2 s, sys: 28.7 s, total: 1min 9s
Wall time: 53.2 s
